## Trabajo Práctico - Estadística Avanzada y Modelos de Regresión
### Análisis de propiedades de Properati | Test de la nueva pipeline
#### Grupo: Aboulafia Gerardo, Barquet Amelie, Lombardo Micaela, Vazquez Agustina
---

In [1]:
# Imports 
#
import numpy as np
import pandas as pd
import re
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import shapely.wkt
import geopandas as gpd
import geopy.distance
from shapely.geometry import Point, Polygon, MultiPolygon
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, KFold, RandomizedSearchCV
from scipy import stats
from shapely.ops import unary_union
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import statsmodels.api as sm
import pickle
from preprocess.filter_data import filter_by_currency_place
from preprocess.regex_extraction import extract_features_regex
from preprocess.geo_validation import validate_geo
from preprocess.subte_distance import calculate_subte_distance
from preprocess.clean_outliers import clean_data_outliers
import warnings
warnings.simplefilter(action='ignore', category=Warning)


/Users/gerardoaboulafia/opt/anaconda3/envs/dhdsblend2021/lib/python3.8/site-packages/geopandas/_compat.py:111: UserWarning: The Shapely GEOS version (3.11.1-CAPI-1.17.1) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  warnings.warn(


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("alejandroczernikier/properati-argentina-dataset")

print("Path to dataset files:", path)

Path to dataset files: /Users/gerardoaboulafia/.cache/kagglehub/datasets/alejandroczernikier/properati-argentina-dataset/versions/1


In [3]:
import os

# Ver qué archivos se descargaron
print(os.listdir(path))

['entrenamiento.csv']


In [4]:
# Cargar datos
data = pd.read_csv(f"{path}/entrenamiento.csv")

In [5]:
# Mostrar información del archivo
print("Data types:"+str(data.dtypes))
print("Data shape:"+str(data.shape))

# Mostrar las primeras filas del archivo
data.head()

Data types:id                   int64
ad_type             object
start_date          object
end_date            object
created_on          object
lat                float64
lon                float64
l1                  object
l2                  object
l3                  object
l4                  object
l5                  object
l6                 float64
rooms              float64
bedrooms           float64
bathrooms          float64
surface_total      float64
surface_covered    float64
currency            object
price_period        object
title               object
description         object
property_type       object
operation_type      object
price              float64
dtype: object
Data shape:(992192, 25)


,id,ad_type,start_date,end_date,created_on,lat,lon,l1,l2,l3,...,bathrooms,surface_total,surface_covered,currency,price_period,title,description,property_type,operation_type,price
0,556713,Propiedad,2019-11-29,9999-12-31,2019-11-29,-58.442399,-34.573623,Argentina,Capital Federal,Colegiales,...,2.0,NaN,NaN,USD,NaN,"Departamento en Venta en Belgrano, Capital fed...","Sup total por escritura: 96,47 m2 (cubiertos: ...",Departamento,Venta,259000.0
1,192912,Propiedad,2020-06-05,2020-06-08,2020-06-05,-58.430493,-34.606620,Argentina,Capital Federal,Almagro,...,2.0,77.0,67.0,USD,NaN,Departamento de 3 ambientes en Venta en Almagro,Excelente departamento de tres ambientes ampli...,Departamento,Venta,235500.0
2,238224,Propiedad,2020-07-01,9999-12-31,2020-07-01,-58.491760,-34.574123,Argentina,Capital Federal,Villa Urquiza,...,1.0,60.0,55.0,USD,NaN,Andonaegui 2600 4° - - Departamento en Venta,Excelente 3 ambientes al frente con balcón. Vi...,Departamento,Venta,175000.0
3,257134,Propiedad,2019-08-17,9999-12-31,2019-08-17,-58.420737,-34.631770,Argentina,Capital Federal,Boedo,...,1.0,74.0,47.0,USD,NaN,PH Venta Boedo 2 amb Patio,Corredor Responsable: MARCELO TRUJILLO - CPI ...,PH,Venta,140000.0
4,521738,Propiedad,2019-08-05,2019-08-31,2019-08-05,-58.429983,-34.607225,Argentina,Capital Federal,Almagro,...,1.0,66.0,64.0,USD,NaN,Venta 3 Ambientes - Almagro - Balcón - Ameniti...,Corredor Responsable: Marcelo Trujillo - CUCIC...,Departamento,Venta,173000.0


In [6]:
data.columns

Index(['id', 'ad_type', 'start_date', 'end_date', 'created_on', 'lat', 'lon',
       'l1', 'l2', 'l3', 'l4', 'l5', 'l6', 'rooms', 'bedrooms', 'bathrooms',
       'surface_total', 'surface_covered', 'currency', 'price_period', 'title',
       'description', 'property_type', 'operation_type', 'price'],
      dtype='object')

In [7]:
# keep only 'lat,'lon', 'l1', 'l2', 'l3','rooms','surface_total', 'currency', 'title', 'description', 'property_type', 'operation_type','price'
data = data[['lat', 'lon', 'l1', 'l2', 'l3','rooms','surface_total', 'currency', 'title', 'description', 'property_type', 'operation_type','price']]

# Swap lat and lon
data = data.rename(columns={'lat': 'temp_lat', 'lon': 'lat'})
data = data.rename(columns={'temp_lat': 'lon'})


In [8]:
# 1. Subetapa de filtrado por currency y l2
data1 = filter_by_currency_place(data)

In [9]:
data1

,lon,lat,l1,l2,l3,rooms,surface_total,currency,title,description,property_type,operation_type,price,flag
0,-58.442399,-34.573623,Argentina,Capital Federal,Colegiales,3.0,NaN,USD,"Departamento en Venta en Belgrano, Capital fed...","Sup total por escritura: 96,47 m2 (cubiertos: ...",Departamento,Venta,259000.0,None
1,-58.430493,-34.606620,Argentina,Capital Federal,Almagro,3.0,77.0,USD,Departamento de 3 ambientes en Venta en Almagro,Excelente departamento de tres ambientes ampli...,Departamento,Venta,235500.0,None
2,-58.491760,-34.574123,Argentina,Capital Federal,Villa Urquiza,2.0,60.0,USD,Andonaegui 2600 4° - - Departamento en Venta,Excelente 3 ambientes al frente con balcón. Vi...,Departamento,Venta,175000.0,None
3,-58.420737,-34.631770,Argentina,Capital Federal,Boedo,2.0,74.0,USD,PH Venta Boedo 2 amb Patio,Corredor Responsable: MARCELO TRUJILLO - CPI ...,PH,Venta,140000.0,None
4,-58.429983,-34.607225,Argentina,Capital Federal,Almagro,3.0,66.0,USD,Venta 3 Ambientes - Almagro - Balcón - Ameniti...,Corredor Responsable: Marcelo Trujillo - CUCIC...,Departamento,Venta,173000.0,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
992187,-57.575169,-38.042062,Argentina,Buenos Aires Costa Atlántica,Mar del Plata,NaN,NaN,USD,Venta en Bloque,Venta en bloque de esta importante propiedad. ...,Casa,Venta,190000.0,Not in Capital Federal
992188,-68.860929,-32.953935,Argentina,Mendoza,Cuadro Benegas,NaN,NaN,USD,PALMARES SEGUNDA ETAPA,RIVEROS PROPIEDADES VENDE: Casa Desarrollada E...,Casa,Venta,520000.0,Not in Capital Federal
992189,-57.963191,-34.920051,Argentina,Bs.As. G.B.A. Zona Sur,La Plata,NaN,NaN,ARS,"Departamento en Alquiler, 44mts, 0 dormitorios...",16 N&deg;706 y 46 se alquila monoambiente e...,Departamento,Alquiler,12000.0,Non-USD currency; Not in Capital Federal
992190,-57.549815,-38.020436,Argentina,Buenos Aires Costa Atlántica,Mar del Plata,NaN,NaN,USD,IMPORTANTE RESIDENCIA EN DIVINO ROSTRO,En importante ubicación del tradicional Barrio...,Casa,Venta,550000.0,Not in Capital Federal


In [10]:
# 2. Subetapa de extracción de valores con RegEx
data2 = extract_features_regex(data1)

In [11]:
data2

,lon,lat,l1,l2,l3,rooms,surface_total,currency,title,description,property_type,operation_type,price,flag,rooms_total,rooms_final,m2_descripcion,m2_final
0,-58.442399,-34.573623,Argentina,Capital Federal,Colegiales,3.0,NaN,USD,"Departamento en Venta en Belgrano, Capital fed...","Sup total por escritura: 96,47 m2 (cubiertos: ...",Departamento,Venta,259000.0,None,NaN,3.0,NaN,NaN
1,-58.430493,-34.606620,Argentina,Capital Federal,Almagro,3.0,77.0,USD,Departamento de 3 ambientes en Venta en Almagro,Excelente departamento de tres ambientes ampli...,Departamento,Venta,235500.0,None,3.0,3.0,NaN,77.0
2,-58.491760,-34.574123,Argentina,Capital Federal,Villa Urquiza,2.0,60.0,USD,Andonaegui 2600 4° - - Departamento en Venta,Excelente 3 ambientes al frente con balcón. Vi...,Departamento,Venta,175000.0,None,NaN,2.0,NaN,60.0
3,-58.420737,-34.631770,Argentina,Capital Federal,Boedo,2.0,74.0,USD,PH Venta Boedo 2 amb Patio,Corredor Responsable: MARCELO TRUJILLO - CPI ...,PH,Venta,140000.0,None,NaN,2.0,NaN,74.0
4,-58.429983,-34.607225,Argentina,Capital Federal,Almagro,3.0,66.0,USD,Venta 3 Ambientes - Almagro - Balcón - Ameniti...,Corredor Responsable: Marcelo Trujillo - CUCIC...,Departamento,Venta,173000.0,None,3.0,3.0,NaN,66.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
992187,-57.575169,-38.042062,Argentina,Buenos Aires Costa Atlántica,Mar del Plata,NaN,NaN,USD,Venta en Bloque,Venta en bloque de esta importante propiedad. ...,Casa,Venta,190000.0,Not in Capital Federal,NaN,NaN,NaN,NaN
992188,-68.860929,-32.953935,Argentina,Mendoza,Cuadro Benegas,NaN,NaN,USD,PALMARES SEGUNDA ETAPA,RIVEROS PROPIEDADES VENDE: Casa Desarrollada E...,Casa,Venta,520000.0,Not in Capital Federal,NaN,NaN,NaN,NaN
992189,-57.963191,-34.920051,Argentina,Bs.As. G.B.A. Zona Sur,La Plata,NaN,NaN,ARS,"Departamento en Alquiler, 44mts, 0 dormitorios...",16 N&deg;706 y 46 se alquila monoambiente e...,Departamento,Alquiler,12000.0,Non-USD currency; Not in Capital Federal,1.0,1.0,44.0,44.0
992190,-57.549815,-38.020436,Argentina,Buenos Aires Costa Atlántica,Mar del Plata,NaN,NaN,USD,IMPORTANTE RESIDENCIA EN DIVINO ROSTRO,En importante ubicación del tradicional Barrio...,Casa,Venta,550000.0,Not in Capital Federal,NaN,NaN,NaN,NaN


In [12]:
# 3. Subetapa de validación geográfica
data3 = validate_geo(data2)

In [13]:
data3

,lon,lat,l1,l2,l3,rooms,surface_total,currency,title,description,property_type,operation_type,price,flag,rooms_total,rooms_final,m2_descripcion,m2_final,en_capital
0,-58.442399,-34.573623,Argentina,Capital Federal,Colegiales,3.0,NaN,USD,"Departamento en Venta en Belgrano, Capital fed...","Sup total por escritura: 96,47 m2 (cubiertos: ...",Departamento,Venta,259000.0,None,NaN,3.0,NaN,NaN,True
1,-58.430493,-34.606620,Argentina,Capital Federal,Almagro,3.0,77.0,USD,Departamento de 3 ambientes en Venta en Almagro,Excelente departamento de tres ambientes ampli...,Departamento,Venta,235500.0,None,3.0,3.0,NaN,77.0,True
2,-58.491760,-34.574123,Argentina,Capital Federal,Villa Urquiza,2.0,60.0,USD,Andonaegui 2600 4° - - Departamento en Venta,Excelente 3 ambientes al frente con balcón. Vi...,Departamento,Venta,175000.0,None,NaN,2.0,NaN,60.0,True
3,-58.420737,-34.631770,Argentina,Capital Federal,Boedo,2.0,74.0,USD,PH Venta Boedo 2 amb Patio,Corredor Responsable: MARCELO TRUJILLO - CPI ...,PH,Venta,140000.0,None,NaN,2.0,NaN,74.0,True
4,-58.429983,-34.607225,Argentina,Capital Federal,Almagro,3.0,66.0,USD,Venta 3 Ambientes - Almagro - Balcón - Ameniti...,Corredor Responsable: Marcelo Trujillo - CUCIC...,Departamento,Venta,173000.0,None,3.0,3.0,NaN,66.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
992187,-57.575169,-38.042062,Argentina,Buenos Aires Costa Atlántica,Mar del Plata,NaN,NaN,USD,Venta en Bloque,Venta en bloque de esta importante propiedad. ...,Casa,Venta,190000.0,Not in Capital FederalNot in Capital Federal (...,NaN,NaN,NaN,NaN,False
992188,-68.860929,-32.953935,Argentina,Mendoza,Cuadro Benegas,NaN,NaN,USD,PALMARES SEGUNDA ETAPA,RIVEROS PROPIEDADES VENDE: Casa Desarrollada E...,Casa,Venta,520000.0,Not in Capital FederalNot in Capital Federal (...,NaN,NaN,NaN,NaN,False
992189,-57.963191,-34.920051,Argentina,Bs.As. G.B.A. Zona Sur,La Plata,NaN,NaN,ARS,"Departamento en Alquiler, 44mts, 0 dormitorios...",16 N&deg;706 y 46 se alquila monoambiente e...,Departamento,Alquiler,12000.0,Non-USD currency; Not in Capital FederalNot in...,1.0,1.0,44.0,44.0,False
992190,-57.549815,-38.020436,Argentina,Buenos Aires Costa Atlántica,Mar del Plata,NaN,NaN,USD,IMPORTANTE RESIDENCIA EN DIVINO ROSTRO,En importante ubicación del tradicional Barrio...,Casa,Venta,550000.0,Not in Capital FederalNot in Capital Federal (...,NaN,NaN,NaN,NaN,False


In [14]:
# 4. Subetapa de cálculo de distancia al subte más cercano
test = data3[:1000]

data4 = calculate_subte_distance(test, '/Users/gerardoaboulafia/Projects/Property_price_prediction/Pipeline/estaciones-de-subte copy.csv')

In [15]:
data4

,lon,lat,l1,l2,l3,rooms,surface_total,currency,title,description,property_type,operation_type,price,flag,rooms_total,rooms_final,m2_descripcion,m2_final,en_capital,distancia_subte_cercano
0,-58.442399,-34.573623,Argentina,Capital Federal,Colegiales,3.0,NaN,USD,"Departamento en Venta en Belgrano, Capital fed...","Sup total por escritura: 96,47 m2 (cubiertos: ...",Departamento,Venta,259000.0,None,NaN,3.0,NaN,NaN,True,0.452056
1,-58.430493,-34.606620,Argentina,Capital Federal,Almagro,3.0,77.0,USD,Departamento de 3 ambientes en Venta en Almagro,Excelente departamento de tres ambientes ampli...,Departamento,Venta,235500.0,None,3.0,3.0,NaN,77.0,True,0.500810
2,-58.491760,-34.574123,Argentina,Capital Federal,Villa Urquiza,2.0,60.0,USD,Andonaegui 2600 4° - - Departamento en Venta,Excelente 3 ambientes al frente con balcón. Vi...,Departamento,Venta,175000.0,None,NaN,2.0,NaN,60.0,True,0.492591
3,-58.420737,-34.631770,Argentina,Capital Federal,Boedo,2.0,74.0,USD,PH Venta Boedo 2 amb Patio,Corredor Responsable: MARCELO TRUJILLO - CPI ...,PH,Venta,140000.0,None,NaN,2.0,NaN,74.0,True,0.765590
4,-58.429983,-34.607225,Argentina,Capital Federal,Almagro,3.0,66.0,USD,Venta 3 Ambientes - Almagro - Balcón - Ameniti...,Corredor Responsable: Marcelo Trujillo - CUCIC...,Departamento,Venta,173000.0,None,3.0,3.0,NaN,66.0,True,0.575219
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,-58.388812,-34.632609,Argentina,Capital Federal,San Cristobal,3.0,66.0,USD,Departamento - San Cristobal - 3 amb - EXCELE...,GRAN OPORTUNIDAD DE INVERSION!!! <br><br>YA R...,Departamento,Venta,120000.0,None,NaN,3.0,NaN,66.0,True,0.873784
996,-58.360168,-34.618946,Argentina,Capital Federal,Puerto Madero,2.0,NaN,USD,Departamento en Venta ubicado en Puerto Mader...,"Departamento de dos dormitorios en suite, sin ...",Departamento,Venta,790000.0,None,NaN,2.0,NaN,NaN,True,1.499044
997,-58.469201,-34.554894,Argentina,Capital Federal,Nuñez,2.0,38.0,USD,Departamento - Nuñez,Hermoso 2 amb. Refaccionado a Nuevo Bajas exp...,Departamento,Venta,90000.0,None,NaN,2.0,NaN,38.0,True,0.630348
998,-58.481682,-34.600352,Argentina,Capital Federal,Villa del Parque,2.0,NaN,USD,Venta excelente piso 2 1/2 amb.!! apto profesi...,"Excelente piso de 2 ambientes y ½, con posibil...",Departamento,Venta,140970.0,None,NaN,2.0,NaN,NaN,True,2.230750


In [16]:
# keep only relevant columns: place_l3	type	rooms_final	m2_final	distancia_subte_cercano price
data4 = data4[['rooms_final', 'm2_final', 'distancia_subte_cercano', 'l3', 'property_type', 'price']]

In [17]:

# 5. Subetapa de limpieza de outliers
print("Limpiando outliers...")
data = clean_data_outliers(data)

Limpiando outliers...


KeyError: 'm2'